In [1]:
#Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
from datasets import Dataset


In [4]:
#We will need the training data text summary pairs
%store -r train_paired_summaries


In [5]:
#looking at data
print(list(train_paired_summaries.keys())[41])

#print(list(train_paired_summaries.values())[41][0]) #text
print(list(train_paired_summaries.values())[41][1]) #summary
print(list(train_paired_summaries.values())[41][2]) #human made flag

Antony_and_Cleopatra-Act II-Scene V
In Alexandria, Cleopatra awaits Antony impatiently. A messenger arrives with news of Antony’s marriage to Octavia. Cleopatra reacts violently, beating the messenger and threatening him. After calming down, she demands more details, insisting on knowing Octavia’s appearance. Her jealousy exposes her vulnerability.
False


In [6]:
#Preprocessing 
#Tokenizing inpts and labels

#is a tupple with (text, summary)
def preprocess_function(batch):
    #add prefix
    input_text = ["summarize: " + text for text in batch["text"]]

    #tokenize text
    model_inputs = t5_tokenizer(
        input_text,
        max_length=128,
        truncation=True,
    )

    #tokenize summary
    with t5_tokenizer.as_target_tokenizer():
        labels = t5_tokenizer(
            batch["summary"],
            max_length=128,
            truncation=True
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
    



In [7]:
#create object to be passed in as batch
#Needs to be dictinionary of lists
#Will be only training data
text_list=[]
summary_list=[]

for v in train_paired_summaries.values():
    text_list.append(v[0])
    summary_list.append(v[1])

training_batch = dict()
training_batch["text"] = text_list
training_batch["summary"] = summary_list

training_batch_dataset = Dataset.from_dict(training_batch)

In [8]:
#Looking at data
print(summary_list[2])
print(list(training_batch.keys()))
print(training_batch["summary"][0])

In the forest, the fairy world is introduced. Oberon and Titania, king and queen of the fairies, argue over a changeling boy whom Titania refuses to give up. Their conflict has disrupted natural order. Oberon plans revenge using a magical flower that makes people fall in love with the first creature they see. Meanwhile, Puck observes Demetrius angrily rejecting Helena, who follows him hopelessly. Oberon pities her and orders Puck to use the magic on Demetrius.
['text', 'summary']
Theseus and Hippolyta plan their upcoming wedding. Egeus arrives with his daughter Hermia, demanding she marry Demetrius instead of Lysander, whom she loves. Theseus upholds Athenian law: Hermia must obey her father or face death or lifelong chastity. After the court leaves, Hermia and Lysander plan to flee into the forest to marry. Helena, secretly in love with Demetrius, learns of their plan and decides to tell Demetrius in hopes of winning his affection.


In [9]:
#Passing data through preprocessor
tokenized_dataset = training_batch_dataset.map(preprocess_function, batched=True, remove_columns=["text", "summary"])

Map:   0%|          | 0/757 [00:00<?, ? examples/s]

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [19]:
#Looking at tokenized data
print(tokenized_dataset[2])
print(len(tokenized_dataset))


{'input_ids': [21603, 10, 10083, 3, 9, 4506, 63, 44, 80, 1365, 6, 11, 276, 4636, 44, 430, 5, 3, 10744, 10459, 5, 571, 230, 6, 3564, 55, 13334, 760, 10735, 25, 58, 377, 18375, 476, 2035, 9956, 6, 147, 3, 5437, 6, 10632, 4607, 17907, 6, 9517, 3, 2160, 49, 6, 2035, 2447, 6, 147, 12674, 6, 10632, 4607, 8347, 6, 9517, 1472, 6, 27, 103, 10735, 6531, 6, 20477, 49, 145, 8, 8114, 22, 7, 3, 9475, 117, 275, 27, 1716, 8, 4506, 63, 5286, 6, 304, 20, 210, 160, 42, 115, 7, 1286, 8, 1442, 5, 37, 9321, 7, 7446, 7, 5065, 160, 8645, 277, 36, 6, 86, 70, 2045, 6001, 7, 6883, 25, 217, 117, 3, 3405, 36, 9641, 725, 6, 19270, 14499, 7, 6, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

Great, at this point its all tokenized

In [21]:
from transformers import Seq2SeqTrainingArguments


In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="t5-small-shakespeare-finetuned",

    do_train=True,
    do_eval=True,

    logging_steps=50,
    save_steps=500,
    eval_steps=500,

    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=3,

    predict_with_generate=True,
    fp16=False,
)

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.